### Import the Libraries and set the style for Plotting

In [ ]:
import torch, torch.nn as nn
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
plt.style.use('dark_background')

### Load the Dataset

##### Training Set
- Input parameters will be:
    - demand at every one of the 1242 junctions
    - pressure at every one of the 1242 junctions
    - flowrate through every one of the 1270 pipes
    - time_of_day

- Therefore the total number of input dimension must be:
$$
1242 + 1242 + 1270 + 1 = 3755
$$
- The output field is leak intensity, i.e. Emitter Coefficients for every leaking node (Last Column of dataset)
- we standardize the variables so that one of them doesn't end up dominating over others when computing the gradients
$$
Z_\text{train}=\frac{X_{\text{train}}-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$
this makes sure that all variables have $\text{mean}=0$ and $\text{standard deviation}=1$

In [ ]:
train_dataset = pd.read_csv(r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset_expanded(train).csv")
X_train = train_dataset.iloc[:, :3755].fillna(0)
train_std = X_train.std()
train_mean = X_train.mean()
X_train = (X_train - train_mean) / train_std
x_train = torch.tensor(X_train.values, dtype=torch.float32)
train_dataset.head()

In [ ]:
y_train = torch.tensor(train_dataset.iloc[:, 3756:])
y_train = y_train.float()

##### Test Set

The training set is what the model learns from. The test data must then be put into the same coordinate system before being given to the trained model. so we will standardize the test set using the mean and variance of the training set and not the test set.
$$
Z_\text{test}=\frac{X_{\text{test}}-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

In [ ]:
test_dataset = pd.read_csv(r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset_expanded(test).csv")
X_test = test_dataset.iloc[:, :3755].fillna(0)
X_test = (X_test - train_mean) / train_std
x_test = torch.tensor(X_test.values, dtype=torch.float32)
test_dataset.head()

In [ ]:
y_test = torch.tensor(test_dataset.iloc[:, 3756:])
y_test = y_train.float()

### Define the Deep Neural Network

In [ ]:
def DNN(D_i, D_k, D_o):
    arch = [D_i] + D_k + [D_o]
    layers = []
    for i in range(len(arch) - 1):
        layers.append(nn.Linear(arch[i], arch[i+1]))
        if i < len(arch) - 2:
            layers.append(nn.ReLU())
    model = nn.Sequential(*layers)
    return model

def network_output(x, model):
    return model(x)

### Define Parameter Initialization

In [ ]:
def initialize_parameters(model):
    for index, layer in enumerate(model):
        if index % 2 == 0:
            nn.init.kaiming_normal_(layer.weight)
            layer.bias.data.fill_(0.0)
        print("Initializing Layer")

### Define Loss Function